# Solutions · Chapter 05-01 · The best constant

Worked answers for `notebooks/05_regression/05-01_baselines.ipynb`.

**E9 and E12 both produce a result that argues against the obvious reading of the chapter**, and they are
the two worth reading even if you got everything else.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# TINY, SYNTHETIC: the chapter's fifteen delivery times
minutes = np.array([22, 25, 27, 28, 30, 31, 33, 34, 36, 38, 41, 44, 47, 52, 58])
candidates = np.linspace(15, 70, 2201)


def pinball(constant, values, q):
    difference = values - constant
    return float(np.mean(np.maximum(q * difference, (q - 1) * difference)))


BASELINE_MAE = float(np.abs(minutes - np.median(minutes)).mean())
BASELINE_MSE = float(((minutes - minutes.mean()) ** 2).mean())
print("baseline: MAE %.4f (at the median %.1f), MSE %.4f (at the mean %.2f)"
      % (BASELINE_MAE, np.median(minutes), BASELINE_MSE, minutes.mean()))

## Quick understanding

### E1

**The median minimises mean absolute error; the mean minimises mean squared error.**

### E2

Because squared error weights an error by its **square**, so a point far from the rest contributes
enormously and drags the minimum towards it - while absolute error weights every point the same, so the
minimum depends only on **how many** points lie either side, not how far away they are.

### E3

**"The delivery time we would beat nine times out of ten."** More precisely, the 90th percentile: the
constant you should promise if arriving later than promised is nine times as costly as arriving earlier.

## Hand calculation

### E4

Numbers 4, 7, 9, 10, 20. **Mean = 50/5 = 10. Median = 9.**

Mean absolute error of each as a constant:

- **Around 10:** `|4-10| + |7-10| + |9-10| + |10-10| + |20-10|` = `6 + 3 + 1 + 0 + 10` = 20, so **4.00**
- **Around 9:** `5 + 2 + 0 + 1 + 11` = 19, so **3.80**

**The median wins**, as it must.

### E5

Mean squared error:

- **Around 10:** `36 + 9 + 1 + 0 + 100` = 146, so **29.20**
- **Around 9:** `25 + 4 + 0 + 1 + 121` = 151, so **30.20**

**The mean wins**, as it must. Each summary wins on its own metric and loses on the other - which is the
whole chapter in five numbers.

### E6

Just **below** 9: two points (4, 7) are below the candidate and three (9, 10, 20) are above, so the slope
is `(2 - 3) / 5` = **-0.20**. The curve is still falling.

Just **above** 9: three points are below and two above, so the slope is `(3 - 2) / 5` = **+0.20**. The
curve is now rising.

**The slope changes sign exactly at 9**, so the minimum is there. That is the general argument: the mean
absolute error falls while more points lie above the candidate than below, and rises once the counts
reverse - so it turns at the value where they balance, which is the definition of the median.

### E7

Unsold loaf costs 1; a turned-away customer costs 4. Being **under** is four times as costly as being
over, so

`q = 4 / (1 + 4)` = **0.80**

The bakery should bake the **80th percentile of demand** - not the average. Baking the mean would leave
it short on roughly half of all days, and on this cost structure each shortfall costs four times what the
waste does.

In [ ]:
small = np.array([4, 7, 9, 10, 20], dtype=float)
print("E4/E5  numbers %s   mean %.1f   median %.1f" % (small.astype(int), small.mean(), np.median(small)))
for label, constant in [("mean", small.mean()), ("median", np.median(small))]:
    print("   as a constant, %-6s = %4.1f  ->  MAE %.2f   MSE %.2f"
          % (label, constant, np.abs(small - constant).mean(), ((small - constant) ** 2).mean()))
print()
for side, below, above in [("just below 9", 2, 3), ("just above 9", 3, 2)]:
    print("E6  %-13s slope = (%d - %d) / 5 = %+.2f" % (side, below, above, (below - above) / 5))
print()
print("E7  q = 4 / (1 + 4) = %.2f  ->  bake the %dth percentile of demand" % (4 / 5, 80))

## Coding

### E8 - a general search

In [ ]:
def best_constant(values, loss, low=None, high=None, points=4001):
    low = values.min() - 1 if low is None else low
    high = values.max() + 1 if high is None else high
    grid = np.linspace(low, high, points)
    losses = np.array([loss(values, c) for c in grid])
    return float(grid[losses.argmin()])


absolute = lambda y, c: np.abs(y - c).mean()
squared = lambda y, c: ((y - c) ** 2).mean()

print("on the fifteen deliveries:")
print("   absolute -> %.3f   (median %.1f)" % (best_constant(minutes, absolute), np.median(minutes)))
print("   squared  -> %.3f   (mean %.4f)" % (best_constant(minutes, squared), minutes.mean()))

skewed = np.random.default_rng(0).lognormal(3.4, 0.35, 200)
print()
print("on 200 draws from a right-skewed distribution:")
print("   absolute -> %.3f   (median %.3f)" % (best_constant(skewed, absolute), np.median(skewed)))
print("   squared  -> %.3f   (mean %.3f)" % (best_constant(skewed, squared), skewed.mean()))

### E9 - how far does each move as the outlier grows?

In [ ]:
rows = []
for outlier in [60, 100, 200, 500, 1000]:
    extended = np.append(minutes, outlier)
    rows.append({"outlier": outlier,
                 "mean moves by": round(extended.mean() - minutes.mean(), 3),
                 "median moves by": round(float(np.median(extended) - np.median(minutes)), 3)})
movement = pd.DataFrame(rows)
print(movement.to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 4.3))
ax.plot(movement.outlier, movement["mean moves by"], "o-", color="#D55E00", linewidth=2.2,
        markersize=8, label="mean")
ax.plot(movement.outlier, movement["median moves by"], "s-", color="#009E73", linewidth=2.2,
        markersize=8, label="median")
ax.set_xlabel("size of the single outlier (minutes)")
ax.set_ylabel("how far the summary moves")
ax.set_title("One is a straight line with slope 1/16. The other is flat", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The median moves exactly 1.00 minute every time - for an outlier of 60 and for an outlier of 1,000.**
The mean's movement is a straight line: 1.475, 3.975, 10.225, 28.975, 60.225, growing in exact proportion
to the outlier.

The shapes are the point, and they are exact rather than approximate:

- **The mean moves linearly.** Adding a value `v` to `n` existing values shifts the mean by
  `(v - old_mean) / (n + 1)`, which is a straight line in `v` with slope `1/(n+1)`. **There is no bound.**
- **The median does not move at all with `v`.** It depends only on the *order*, and once the new value is
  above every existing one, making it larger changes no ordering. Going from 15 to 16 values shifts the
  median from the 8th value to the average of the 8th and 9th, which is the whole 1.00-minute effect -
  and it would be the same if the outlier were a million.

**"Robust" is not vague praise; it is this graph.** A single arbitrarily bad value can move a mean
arbitrarily far and cannot move a median past its neighbour.

### E10 - going further than squaring

In [ ]:
cubed = best_constant(minutes, lambda y, c: np.mean(np.abs(y - c) ** 3), 15, 70, 5501)
print("best constant under |error|^1  (absolute): %.2f   <- the median" % np.median(minutes))
print("best constant under |error|^2  (squared) : %.2f   <- the mean" % minutes.mean())
print("best constant under |error|^3            : %.2f" % cubed)
print()
print("each step of the exponent pulls the answer further into the right tail")

**37.68 - past the mean, further into the tail.**

The progression is the chapter's scale made explicit: **the higher the power, the more the biggest errors
dominate, and the further the optimal constant is dragged towards them.** At power 1 the answer ignores
distances entirely (the median); at power 2 it balances them; at power 3 the largest errors are worth so
much that it is worth accepting many small ones to shrink one big one. In the limit of a very high power
you approach the value that minimises the *worst* error, which is the midpoint of the range.

Practically: **the exponent is a statement about how costs grow with the size of the mistake**, and it
does not have to be 1 or 2. It is 2 by convention and because squared error is differentiable, not
because being wrong usually costs quadratically.

### E11 - scoring candidate constants

In [ ]:
def skill(baseline_error, model_error):
    return (baseline_error - model_error) / baseline_error


rows = []
for constant in [30, 34, 40]:
    mae = float(np.abs(minutes - constant).mean())
    mse = float(((minutes - constant) ** 2).mean())
    rows.append({"constant": constant,
                 "MAE": round(mae, 3), "skill vs 8.00": "%+.2f%%" % (100 * skill(BASELINE_MAE, mae)),
                 "MSE": round(mse, 3), "skill vs 99.17": "%+.2f%%" % (100 * skill(BASELINE_MSE, mse))})
print(pd.DataFrame(rows).to_string(index=False))

**Which looks worst depends entirely on the metric.**

- Judged on **absolute** error, **40 is worst** (-15.00%) and 30 is second (-10.00%).
- Judged on **squared** error, **30 is worst** (-41.30%) and 40 is second (-13.07%).

The ranking reverses. 30 is further from the slow deliveries in the tail, and squaring makes those
distances dominate; 40 is further from the many fast ones, which absolute error counts equally.

Note also that **34 scores exactly 0% skill on MAE** - it *is* the MAE baseline - while scoring -5.81% on
MSE. A predictor that is optimal under one metric is a below-baseline predictor under another, on the same
data, with no modelling involved at all.

### E12 - which summary is more reliable?

The chapter argued for the median's robustness. **Predict first:** which of the two varies more from
sample to sample?

In [ ]:
sampler = np.random.default_rng(0)
samples = sampler.lognormal(3.4, 0.35, size=(1000, 15))
sample_means = samples.mean(axis=1)
sample_medians = np.median(samples, axis=1)

print("1,000 samples of 15 values each, from a right-skewed distribution:")
print("   sample means  : sd %.4f" % sample_means.std())
print("   sample medians: sd %.4f" % sample_medians.std())
print("   the median is %.2f times as variable" % (sample_medians.std() / sample_means.std()))

# centre each on its own average, so the picture compares spread and nothing else
centred_means = sample_means - sample_means.mean()
centred_medians = sample_medians - sample_medians.mean()

fig, ax = plt.subplots(figsize=(9.5, 4.4))
edges = np.linspace(min(centred_means.min(), centred_medians.min()),
                    max(centred_means.max(), centred_medians.max()), 55)
ax.hist(centred_means, bins=edges, alpha=0.7, color="#D55E00",
        label="sample mean   (sd %.3f)" % sample_means.std())
ax.hist(centred_medians, bins=edges, alpha=0.7, color="#009E73",
        label="sample median (sd %.3f)" % sample_medians.std())
for spread, colour in [(sample_means.std(), "#D55E00"), (sample_medians.std(), "#009E73")]:
    ax.axvline(spread, color=colour, linestyle="--", linewidth=1.8)
    ax.axvline(-spread, color=colour, linestyle="--", linewidth=1.8)
ax.set_xlabel("distance from that summary's own average")
ax.set_ylabel("samples")
ax.set_title("Both centred: the robust one is the wider one", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The median is 1.15 times as variable as the mean**, which is the opposite of what "robust" suggests to
most people.

Both facts are true and they are about different things:

- **Robustness** is about a *single contaminated value*. One absurd observation moves the mean without
  limit and the median barely at all - E9.
- **Efficiency** is about *ordinary sampling noise*. The mean uses every observation's value; the median
  uses essentially the middle one or two. With clean data from a mildly skewed distribution, throwing
  away that information costs precision.

**So the advice from E1 does not change, but its justification does.** Use the median when the metric is
absolute error - because it is optimal for that metric, which is a fact about the loss, not about
robustness. Use it *additionally* when you expect contaminated data. **Do not** reach for it under a
vague belief that it is the safer summary in general, because on clean data it is the noisier one.

(For heavy-tailed distributions the comparison flips and the median becomes the more precise estimator
too. Which is exactly the point: it depends on the data, and the check is the four lines above.)

## Interpretation

### E13

**Reason 1: 36 minutes is the answer to a question nobody asked.** The mean is optimal under *squared*
error, which says a 20-minute miss is sixteen times as bad as a 5-minute one. A customer promise is not
scored that way - the cost of being late is roughly proportional to how late, or worse, has a cliff at
the promised time.

**Reason 2: a promise is asymmetric, and the mean assumes it is not.** Being early is nearly free; being
late costs a complaint. That is a pinball problem, not a mean problem - and the constant it asks for is a
high quantile, well above both 34 and 36.4. On this data the 90th percentile is **52 minutes**.

The deeper mistake is treating "what is the average?" as the same question as "what should we promise?".
The average is a description of the data. **The promise is a decision, and decisions have costs
attached.**

### E14

`(8.00 - 7.20) / 8.00` = **10.0% skill** against the median baseline.

**Whether that is good depends on what 0.8 minutes buys.** It is a real improvement - the model is not
just re-deriving the constant - but it is modest, and on fifteen deliveries it is well within what a
different sample would have produced. Before calling it good I would want the spread across splits
(04-03), and I would want to know whether 48 seconds of average accuracy changes any decision.

The honest sentence: **"10% better than predicting the median, measured on fifteen deliveries, which is
too few to be confident about."**

### E15

**8.4 is worse than the baseline, not close to it.** The baseline is 8.00, so their skill is
`(8.00 - 8.40) / 8.00` = **-5%**.

"Close to the baseline" describes the *number*; the sign is what matters. A model that costs a
development cycle and performs worse than a single constant should not ship, and calling it "close"
obscures the only fact that decides that.

The likely underlying error is comparing against the wrong baseline - probably the *mean* constant, which
scores 8.21 under MAE, and against which 8.4 is indeed only slightly worse. **Quoting the weaker baseline
is 04-02's warning**, and here it converts a negative result into a neutral-sounding one.

## Debugging

### E16

**The mechanism:** MAPE divides each error by the actual value. An under-prediction on a small actual
produces a large percentage; an over-prediction on a large actual produces a small one. The cheapest way
to keep the average percentage low is therefore to **predict low**, and a system tuned on MAPE learns
exactly that. On this chapter's data MAPE's best constant is 31.0 minutes against a median of 34.

Under-ordering stock is the direct consequence: the forecast is systematically below demand, because the
metric rewarded it.

**A metric that would not do this:** mean absolute error, if the cost of being wrong is proportional to
the amount. Better still for stock, **pinball loss at a `q` set from the actual costs** - if a stockout
costs three times what waste costs, `q = 0.75`. That fixes the direction of the bias *and* points it the
right way, which MAE alone does not.

If a scale-free metric is genuinely needed - comparing across products of very different volumes -
**weighted absolute error** (total absolute error divided by total actual) avoids the worst of MAPE's
behaviour, because it does not divide each error by its own actual.

## Exam and interview reasoning

### E17

> "A constant. Specifically, the constant that is optimal for whatever metric I am being judged on: the
> median if it is absolute error, the mean if it is squared error, the appropriate quantile if the costs
> are asymmetric. That gives a number in the units of the target, so 'the model must beat 8 minutes of
> average error' is something a stakeholder can judge. If rows have a natural grouping I would also add a
> per-group constant, because that is often much harder to beat - in module 04 a per-member average beat
> both a linear regression and a random forest."

**"Why not just always use the mean?"**

> "Because it is only the best constant if I am being scored on squared error. If the metric is absolute
> error, the mean is a *worse* baseline than the median, so quoting it makes my model look better than it
> is - on this data the mean scores 8.21 against the median's 8.00. And the mean is easy to move: in the
> delivery example one stuck driver in sixteen shifted it by ten minutes while the median moved by one.
> If I quote a baseline, I want it to be the strongest one I can find, not the most convenient."

## Transfer to a different situation

### E18

**The metric: pinball loss**, with `q` set from the staffing costs. If being one person short costs three
times what having one spare costs, `q = 0.75`.

**The baseline it implies:** the **75th percentile** of historical resolution times - not the mean, and
not the median. Concretely: sort past tickets by resolution time, take the value three-quarters of the way
up, and staff to that.

**What to tell the team**, and this is the part that matters more than the arithmetic:

> "This number is deliberately not the average. It is the level we expect to be enough three times out of
> four, because you told me being short is three times worse than being over. If that ratio is wrong, tell
> me and the number changes - it is the only input."

Two things to add if pressed: the metric should be recomputed by segment if some ticket types behave
differently (05-12), and the `q` should be revisited when the cost ratio changes, because it is the *only*
thing that determines it.

## Explain it to someone non-technical

### E19

> "Say deliveries usually take about half an hour, but a few take nearly an hour. If I ask 'what one
> number best describes these?', the answer depends on what a wrong guess costs you. If being off by ten
> minutes is twice as bad as being off by five, one number is best. If it is four times as bad, a higher
> number is best, because the slow deliveries matter more. And if being late upsets customers but being
> early does not, a higher number again. Same deliveries, three different right answers."

(88 words.)

## Optional challenge

### E20 - the ceiling rule

In [ ]:
rows = []
for n in range(5, 41):
    sample = np.sort(np.random.default_rng(n).lognormal(3.4, 0.4, n))
    grid = np.linspace(sample.min() - 1, sample.max() + 1, 8001)
    for q in [0.1, 0.3, 0.5, 0.7, 0.9]:
        losses = np.array([pinball(c, sample, q) for c in grid])
        rule_value = sample[int(np.ceil(q * n)) - 1]
        rows.append({
            "n": n, "q": q,
            # is the rule's value the SAME POINT the search found?
            "same point": bool(abs(grid[losses.argmin()] - rule_value) < 0.02),
            # and, the question that actually matters, is it optimal?
            "optimal": bool(pinball(rule_value, sample, q) <= losses.min() + 1e-9),
            "q x n is an integer": bool(abs(q * n - round(q * n)) < 1e-9)})

check = pd.DataFrame(rows)
print("cases tried: %d" % len(check))
print("  the rule's value is OPTIMAL in            : %d  (%.1f%%)"
      % (check.optimal.sum(), 100 * check.optimal.mean()))
print("  it is the same point the search found in  : %d  (%.1f%%)"
      % (check["same point"].sum(), 100 * check["same point"].mean()))
print()
disagreements = check[~check["same point"]]
print("the %d disagreements, and whether q x n was an integer in each:" % len(disagreements))
print(disagreements["q x n is an integer"].value_counts().to_string())

**The rule's value is optimal in all 180 cases**, and it is the point the search returns in 171 of them.

**All nine disagreements have `q x n` an exact integer** - and in every one, the loss at the rule's value
equals the minimum. So the rule never gives a wrong answer; in those nine it gives one of several equally
correct ones, and `argmin` happened to return a different member of the same flat region.

**Why a ceiling rather than rounding.** Walk the candidate constant upward from below the data. Between
two neighbouring observations, the slope of the pinball loss is constant, and it equals

`(1 - q) x (number of points at or below) - q x (number of points above)`, divided by `n`.

The loss falls while that expression is negative and rises once it turns positive, so the minimum is at
the first observation where the count at or below reaches `q x n`. **"The first integer that reaches
`q x n`" is precisely the ceiling** - if `q x n` is 11.25, then eleven points are not yet enough and
twelve are, so the twelfth smallest is the answer.

Rounding would give eleven whenever the fractional part is below a half, which is wrong for exactly the
cases the ceiling exists to handle.

**Which is exactly what the nine disagreements are.** When `q x n` is an integer the slope between two
observations is zero, so every constant in that interval is optimal and the minimiser is not unique -
predicted by the formula above, and confirmed by the fact that all nine, and only those nine, have an
integer `q x n`. `np.quantile` resolves the ambiguity by interpolating; the grid search returns whichever
point it met first. Neither is more correct - **the sample simply does not pin the quantile down.**

That is a small result, but it is the kind worth deriving once: **it explains why two library functions
that both claim to compute "the quantile" can disagree, and why neither is a bug.**